# Predictive Anayltics: Support Vector Machines

Approach for SVM:
– Simply start without a kernel. Then, gradually make your model complex by integrating different
kind of kernels. Also, use grid search to find optimal values for your hyperparameters.
– How good is your model? Evaluate your model’s performance and comment on its shortfalls.
– Show how you model’s performance varies as you increase or decrease temporal or spatial
resolution How does your performance change when you only use census tract as spatial units?
– How could the model be improved further? Explain some of the improvement levers that you might
focus on in a follow-up project.

In [1]:
import pandas as pd
import polars as pl
import numpy as np
import matplotlib as plt
import datetime
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV

## Preparations

In [2]:
# "Settings" / Decisions for the training data

DATA_PATH_TRAIN = "../data/train_test_data/train.parquet"
DATA_PATH_VAL = "../data/train_test_data/val.parquet"
DATA_PATH_TEST = "../data/train_test_data/test.parquet"

MODEL_PATH = "../models"

# Target and feature selection
TARGET_COL = "trip_count"
EXCLUDE_COLS = [
    TARGET_COL,
    "datetime_hour",      # real timestamp would lead to much leakage
    "_split_bucket",      # only used for splitting
    "month", # cyclic feature is used instead
    "weekday", # cyclic feature is used instead
    "hour", # cyclic feature is used instead
    # all other taxi data columns must be excluded to prevent leakage
    "trip_seconds_sum",
    "trip_seconds_mean",
    "trip_seconds_min",
    "trip_seconds_max",
    "trip_miles_sum",
    "trip_miles_mean",
    "trip_miles_min",
    "trip_miles_max",
    "fare_sum",
    "fare_mean",
    "fare_min",
    "fare_max",
    "tips_sum",
    "tips_mean",
    "tips_min",
    "tips_max",
    "tolls_sum",
    "tolls_mean",
    "tolls_min",
    "tolls_max",
    "extras_sum",
    "extras_mean",
    "extras_min",
    "extras_max",
    "trip_total_sum",
    "trip_total_mean",
    "trip_total_min",
    "trip_total_max",
    "most_common_payment_type",
    "trip_demand",
    "date",
]

Load data and select features and target

In [3]:
# Load data
train = pl.scan_parquet(DATA_PATH_TRAIN)
val = pl.scan_parquet(DATA_PATH_VAL)
test = pl.scan_parquet(DATA_PATH_TEST)

In [4]:
train_df = train.collect()
val_df = val.collect()
test_df = test.collect()

train_df = train_df.to_pandas()
val_df = val_df.to_pandas()
test_df = test_df.to_pandas()

In [5]:
train_df.head()
type(train_df)

pandas.DataFrame

In [6]:
# prepare data
# calculate median to split in low/high demand
# when trip_count above 50 percent use "high", when below or equal to 50 percent low
train_median = train_df["trip_count"].median()
train_df["trip_demand"] = np.where(train_df["trip_count"] > train_median, "high", "low")

val_median = val_df["trip_count"].median()
val_df["trip_demand"] = np.where(val_df["trip_count"] > val_median, "high", "low")

test_median = test_df["trip_count"].median()
test_df["trip_demand"] = np.where(test_df["trip_count"] > test_median, "high", "low")

In [7]:
train_df = train_df.sample(n=500, random_state=42)

In [8]:
# Create X and y
feature_cols = [
    col for col in train_df.columns
    if col not in EXCLUDE_COLS
]

X_train = train_df[feature_cols]
y_train = train_df[TARGET_COL]

X_val = val_df[feature_cols]
y_val = val_df[TARGET_COL]

X_test = test_df[feature_cols]
y_test = test_df[TARGET_COL]


print("Features:", X_train.dtypes)
print("Target:", y_train.dtypes)

Features: month_sin         float64
month_cos         float64
weekday_sin       float64
weekday_cos       float64
hour_sin          float64
hour_cos          float64
tmpc              float64
relh              float64
sknt              float64
vsby              float64
p01m              float64
skyc1_BKN            int8
skyc1_CLR            int8
skyc1_FEW            int8
skyc1_OVC            int8
skyc1_SCT            int8
skyc1_VV             int8
is_holiday           int8
community_area      int64
food_drink        float64
landmark          float64
shop              float64
train_station     float64
dtype: object
Target: uint32


In [9]:
train_df

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type,trip_demand
12153,2024-03-15 09:00:00,3,5,9,0.866025,5.000000e-01,-0.433884,-0.900969,7.071068e-01,-7.071068e-01,...,0.0,0.000000,0.0,0.0,37.60,37.600000,37.60,37.60,Prcard,low
653944,2024-08-18 20:00:00,8,7,20,-0.500000,-8.660254e-01,-0.781831,0.623490,-8.660254e-01,5.000000e-01,...,0.0,0.000000,0.0,0.0,0.00,0.000000,0.00,0.00,No trips,low
1073833,2024-05-22 18:00:00,5,3,18,0.866025,-5.000000e-01,0.974928,-0.222521,-1.000000e+00,-1.836970e-16,...,0.0,0.000000,0.0,0.0,0.00,0.000000,0.00,0.00,No trips,low
705090,2025-05-13 04:00:00,5,2,4,0.866025,-5.000000e-01,0.781831,0.623490,8.660254e-01,5.000000e-01,...,0.0,0.000000,0.0,0.0,72.68,24.226667,10.47,46.83,Mobile,high
982471,2025-04-10 12:00:00,4,4,12,1.000000,6.123234e-17,0.433884,-0.900969,1.224647e-16,-1.000000e+00,...,0.0,0.000000,0.0,0.0,0.00,0.000000,0.00,0.00,No trips,low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
818913,2025-03-10 11:00:00,3,1,11,0.866025,5.000000e-01,0.000000,1.000000,2.588190e-01,-9.659258e-01,...,1.0,0.090909,0.0,1.0,263.09,23.917273,7.75,31.00,Prcard,high
1061565,2025-08-09 15:00:00,8,6,15,-0.500000,-8.660254e-01,-0.974928,-0.222521,-7.071068e-01,-7.071068e-01,...,0.0,0.000000,0.0,0.0,60.75,30.375000,30.25,30.50,Unknown,high
795342,2025-01-31 22:00:00,1,5,22,0.000000,1.000000e+00,-0.433884,-0.900969,-5.000000e-01,8.660254e-01,...,0.0,0.000000,0.0,0.0,0.00,0.000000,0.00,0.00,No trips,low
887040,2024-10-19 00:00:00,10,6,0,-1.000000,-1.836970e-16,-0.974928,-0.222521,0.000000e+00,1.000000e+00,...,0.0,0.000000,0.0,0.0,30.00,30.000000,30.00,30.00,Prcard,low


In [10]:
model = SVR()

In [ ]:
param_grid = {
    "C": [1, 10, 100],
    "kernel": ["linear", "rbf", "poly", "sigmoid"],
    "gamma": ["scale", "auto", 0.01, 0.1, 1]
}

# Perform grid search
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,               
    scoring="r2",  
    n_jobs=-1,
    error_score="raise"
)

# Fit
grid_search.fit(X_train, y_train)


In [ ]:
# Best parameters
print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

Best parameters: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
Best CV score: 0.15290691677835486


In [ ]:
# Train SVC 

model.fit(X_train, y_train)

,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,tol,0.001
,C,1.0
,epsilon,0.1
,shrinking,True
,cache_size,200
,verbose,False
,max_iter,-1


In [ ]:
# Make prediction 
y_pred = model.predict(X_test)

In [ ]:
y_pred

array([0.86370242, 1.03050174, 1.12443349, ..., 0.58577163, 0.39647657,
       0.71869768], shape=(223531,))

In [ ]:
# Evaluation metrics

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

MAE: 7.9992880345730235
MSE: 1226.8952250191385
RMSE: 35.027064179276266
R2 Score: 0.0077389154144491545
